# Silver Layer — Electronic Medication Administration Details (`emar_detail`)

**Source:** `<source_schema>.emar_detail` (default `cdr_mimiciv.bronze.emar_detail`)  
**Target:** `<target_schema>.emar_detail` (default `cdr_mimiciv.silver.emar_detail`)

**Objective:** Standardize detailed electronic medication administration records while preserving medication administration identifiers, dose and route information and administration details.

**Grain:** One source eMAR detail record (MIMIC-IV hosp `emar_detail`: detail information for a medication administration in `emar`, linked on `emar_id`; `subject_id`, `emar_id` and `emar_seq` are documented NOT NULL). Multiple detail records may belong to the same medication administration event, so `emar_id` is not a unique key and 4,861 exact duplicate groups exist in the source (below). No unique key is established: uniqueness is **profiled only, not asserted**.

**Silver rules:** Preserve every source record and the 33 documented clinical columns with their Bronze types. Bronze columns beyond the documentation are excluded from Silver until they are identified and a decision is recorded. Do not delete, filter, deduplicate, impute, aggregate, or automatically correct records.

**Data quality:** Inspect missing values, exact duplicates, identifiers, administration details, and Bronze–Silver row counts. Document anomalies without altering source data.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped. Hardcoded profiled counts are asserted only when widget `enforce_profiled_counts` is `true`.

**Source-data quality finding:** 4,861 exact duplicate groups were
identified across all 36 columns of the profiled Bronze table. The official documentation and the local source
CSV header list 33 columns; the three additional Bronze columns were not identified. A re-derivation from the
local source CSV (87,371,064 rows) found no byte-identical duplicate rows. A **local simulation of typing**
(parsing numeric-looking text as floating point in Python; this is *not* verified Spark behaviour) gives
exactly the same figures: 4,861 groups, 9,722 records, maximum group size 2. This suggests the groups are rows
that become identical once numeric text is typed rather than repeated source lines, but it must be confirmed
against the real Bronze table (see `docs/gold_governance_checks.md`, Source readiness). Duplicates are preserved. Each group contains two records,
representing 9,722 records in duplicate groups and 4,861 additional
duplicate occurrences. All records are retained without deduplication.

**Missing-value findings:** The source contains 42,808,593 NULL
`parent_field_ordinal` values (documented as structural: there is one NULL-ordinal row per
eMAR order, usually holding the dose due, followed by rows `1.1`, `1.2`, ... for formulary
doses), 44,569,078 NULL `administration_type`
values, 45,761,669 NULL `dose_due` values, 44,955,793 NULL
`dose_due_unit` values, and 46,269,155 NULL `dose_given` values.
All missing values are retained without imputation or record removal.

**Cost note:** very large table (tens of millions of rows). Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

SOURCE_TABLE = f"{BRONZE}.emar_detail"
TARGET_TABLE = f"{SILVER}.emar_detail"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

In [0]:
# Read all eMAR detail records without filtering or modifying data
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

In [0]:
# Inspect the actual Bronze schema before choosing Silver conversions
schema_df = spark.createDataFrame(
    [(field.name, field.dataType.simpleString(), field.nullable)
     for field in bronze_df.schema.fields],
    ["column_name", "bronze_type", "nullable"]
)

display(schema_df)

In [0]:
# Preserve every eMAR detail record and the 33 documented clinical columns. Bronze columns beyond the
# documentation are NOT passed into Silver automatically: they are listed here and must be identified
# (and a decision recorded) before any is carried forward.
DOCUMENTED_COLUMNS = [
    "subject_id", "emar_id", "emar_seq", "parent_field_ordinal", "administration_type", "pharmacy_id",
    "barcode_type", "reason_for_no_barcode", "complete_dose_not_given", "dose_due", "dose_due_unit",
    "dose_given", "dose_given_unit", "will_remainder_of_dose_be_given", "product_amount_given",
    "product_unit", "product_code", "product_description", "product_description_other",
    "prior_infusion_rate", "infusion_rate", "infusion_rate_adjustment", "infusion_rate_adjustment_amount",
    "infusion_rate_unit", "route", "infusion_complete", "completion_interval", "new_iv_bag_hung",
    "continued_infusion_in_other_location", "restart_interval", "side", "site",
    "non_formulary_visual_verification",
]

missing_documented = [c for c in DOCUMENTED_COLUMNS if c not in bronze_df.columns]
assert not missing_documented, f"Documented columns missing from Bronze: {missing_documented}"
EXTRA_BRONZE_COLUMNS = [c for c in bronze_df.columns if c not in DOCUMENTED_COLUMNS]
print("Bronze columns not in the documentation (excluded from Silver, to be identified):", EXTRA_BRONZE_COLUMNS)

silver_df = bronze_df.select(*DOCUMENTED_COLUMNS)

silver_df.printSchema()

# Ordered expected schema (the 33 documented columns, with their Bronze types)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}

In [0]:
# Single pass: row counts, missingness for every column and dose/administration fields
bronze_count = bronze_df.count()

critical_columns = [c for c in ["subject_id", "emar_id", "emar_seq"] if c in silver_df.columns]

aggs = [F.count("*").alias("total_records")]
aggs += [
    F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
    for c in silver_df.columns
]
profile = silver_df.agg(*aggs).first().asDict()
display(spark.createDataFrame([(k, int(v or 0)) for k, v in profile.items()], ["metric", "value"]))

silver_count = profile["total_records"]

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

In [0]:
# Identify exact duplicate records without removing them (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("additional_duplicate_occurrences"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate eMAR detail groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])
print("Additional duplicate occurrences:", dup_row["additional_duplicate_occurrences"])
print("Maximum group size:", dup_row["maximum_group_size"])

In [0]:
# Pre-publish validation
critical_nulls = {f"null_{c}": profile[f"null_{c}"] for c in critical_columns}
print("Critical NULL counts:", critical_nulls)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert silver_df.schema == bronze_df.select(*DOCUMENTED_COLUMNS).schema, "Documented column types changed"
assert all(v == 0 for v in critical_nulls.values()), (
    f"Documented NOT NULL identifiers contain NULLs: {critical_nulls}"
)

if ENFORCE_PROFILED_COUNTS:
    assert duplicate_count == 4861, "Duplicate group count changed; review before publication"
    assert dup_row["records_in_duplicate_groups"] == 9722, "Records in duplicate groups changed"
    assert dup_row["additional_duplicate_occurrences"] == 4861, "Additional duplicate occurrences changed"
    assert dup_row["maximum_group_size"] == 2, "Maximum duplicate group size changed"
    expected_nulls = {
        "parent_field_ordinal": 42808593,
        "administration_type": 44569078,
        "dose_due": 45761669,
        "dose_due_unit": 44955793,
        "dose_given": 46269155,
    }
    for _c, _n in expected_nulls.items():
        assert profile[f"null_{_c}"] == _n, f"null {_c} count changed: {profile[f'null_{_c}']}"

print("All pre-publish checks passed.")

In [0]:
%sql
drop table cdr_mimiciv.silver.emar_detail;

In [0]:
# Publish all eMAR detail records (CREATE OR REPLACE), including documented exact duplicates
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

In [0]:
# Verify published eMAR detail row count, schema and identifiers (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in critical_columns]
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert all(v == 0 for k, v in published_stats.items() if k.startswith("null_")), (
    "Published identifiers contain NULLs"
)

print("Silver eMAR detail table verified successfully.")

In [0]:
dbutils.notebook.exit("emar_detail:OK")